## Resume Scanner
Owen Lee and Ben Schrager

# SWE Resume Reviewer

This project analyzes a software engineering applicant dataset to predict the likelihood that an applicant is hired.

## Research Question

Can we use applicant features to predict the likelihood that a software engineering applicant is hired?

## Supporting Questions

1. Which applicant features matter most in predicting hiring outcomes?

2. Can we turn the prediction into a simple resume-style evaluator for users?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('stackoverflow_full.csv')
df = df.rename(columns={"Unnamed: 0": "ApplicantID"})
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Basic EDA
print("Target variable distribution:")
print(df["Employed"].value_counts())
print("\nTarget variable proportions:")
print(df["Employed"].value_counts(normalize=True))

# Define feature groups
target = "Employed"
id_cols = ["ApplicantID"]
numeric_cols = ["Employment", "YearsCode", "YearsCodePro", "PreviousSalary"]
categorical_cols = ["Age", "Accessibility", "EdLevel", "Gender", "MentalHealth", "MainBranch", "Country"]
text_cols = ["HaveWorkedWith"]

## Random Forest Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Baseline features
baseline_features = [
    "EdLevel",
    "MainBranch",
    "YearsCode",
    "YearsCodePro",
    "PreviousSalary"
]

X = df[baseline_features].copy()
y = df[target].copy()

categorical_features = ["EdLevel", "MainBranch"]
numeric_features = ["YearsCode", "YearsCodePro", "PreviousSalary"]

# Preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Random Forest model
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

# Predictions
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

print("=== Random Forest Results ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Feature importance analysis
feature_names = rf_model.named_steps["preprocessor"].get_feature_names_out()
importances = rf_model.named_steps["classifier"].feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("Feature Importances:")
print(feature_importance_df.head(15))

In [ ]:
# Visualize feature importance
top_n = min(15, len(feature_importance_df))
top_features = feature_importance_df.head(top_n)

plt.figure(figsize=(10, 6))
plt.barh(top_features["Feature"][::-1], top_features["Importance"][::-1])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importances")
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix visualization
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Random Forest")
plt.show()

Our random forest model demonstrates solid predictive performance using key applicant characteristics. The model relies most heavily on experience metrics (YearsCode, YearsCodePro) and PreviousSalary, while education level and professional status contribute to a lesser extent.

In [ ]:
# Check for overfitting
y_train_pred = rf_model.predict(X_train)
y_train_proba = rf_model.predict_proba(X_train)[:, 1]

print("=== Random Forest: Train vs Test Performance ===")
print("TRAIN Accuracy:", accuracy_score(y_train, y_train_pred))
print("TRAIN F1:", f1_score(y_train, y_train_pred))
print("TRAIN ROC-AUC:", roc_auc_score(y_train, y_train_proba))
print()
print("TEST Accuracy:", accuracy_score(y_test, y_pred))
print("TEST F1:", f1_score(y_test, y_pred))
print("TEST ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

## Logistic Regression

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Baseline features
baseline_features = [
    "EdLevel",
    "MainBranch",
    "YearsCode",
    "YearsCodePro",
    "PreviousSalary"
]

X = df[baseline_features].copy()
y = df["Employed"].copy()

categorical_features = ["EdLevel", "MainBranch"]
numeric_features = ["YearsCode", "YearsCodePro", "PreviousSalary"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

log_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42))
])

log_model.fit(X_train, y_train)

y_train_pred = log_model.predict(X_train)
y_train_proba = log_model.predict_proba(X_train)[:, 1]

y_test_pred = log_model.predict(X_test)
y_test_proba = log_model.predict_proba(X_test)[:, 1]

print("=== Logistic Regression Results ===")
print("\nTRAIN Performance:")
print("TRAIN Accuracy:", accuracy_score(y_train, y_train_pred))
print("TRAIN F1:", f1_score(y_train, y_train_pred))
print("TRAIN ROC-AUC:", roc_auc_score(y_train, y_train_proba))

print("\nTEST Performance:")
print("TEST Accuracy:", accuracy_score(y_test, y_test_pred))
print("TEST Precision:", precision_score(y_test, y_test_pred))
print("TEST Recall:", recall_score(y_test, y_test_pred))
print("TEST F1:", f1_score(y_test, y_test_pred))
print("TEST ROC-AUC:", roc_auc_score(y_test, y_test_proba))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

We compared random forest and logistic regression models on our baseline feature set. While random forest shows strong training performance, logistic regression demonstrates better generalization with more consistent train-test performance, making it our preferred model for interpretability and reliability.

## SHAP Analysis

In [ ]:
import shap

# Extract the preprocessed feature matrix
X_test_transformed = log_model.named_steps["preprocessor"].transform(X_test)
feature_names_out = log_model.named_steps["preprocessor"].get_feature_names_out()

# Build a SHAP explainer around the fitted logistic regression
explainer = shap.LinearExplainer(
    log_model.named_steps["classifier"],
    X_test_transformed,
    feature_perturbation="interventional"
)
shap_values = explainer.shap_values(X_test_transformed)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Number of features: {len(feature_names_out)}")

In [ ]:
# Summary plot: global feature importance + direction
plt.figure()
shap.summary_plot(
    shap_values,
    X_test_transformed,
    feature_names=feature_names_out,
    show=False
)
plt.title("SHAP Summary Plot — Logistic Regression")
plt.tight_layout()
plt.show()

In [ ]:
# Bar plot: mean absolute SHAP value per feature
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance_df = pd.DataFrame({
    "Feature": feature_names_out,
    "Mean |SHAP|": mean_abs_shap
}).sort_values("Mean |SHAP|", ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(shap_importance_df["Feature"][::-1], shap_importance_df["Mean |SHAP|"][::-1], color="steelblue")
plt.xlabel("Mean |SHAP Value|")
plt.title("Feature Importance by Mean Absolute SHAP Value")
plt.tight_layout()
plt.show()

print("\nFeature Importance (SHAP):")
print(shap_importance_df.to_string(index=False))

In [ ]:
# Waterfall plots: explain two individual predictions
y_pred_test = log_model.predict(X_test)

hired_idx = np.where(y_pred_test == 1)[0][0]   # first predicted-hired
rejected_idx = np.where(y_pred_test == 0)[0][0]  # first predicted-not-hired

base_value = explainer.expected_value

for idx, label in [(hired_idx, "Predicted: HIRED"), (rejected_idx, "Predicted: NOT HIRED")]:
    exp = shap.Explanation(
        values=shap_values[idx],
        base_values=base_value,
        data=X_test_transformed[idx],
        feature_names=list(feature_names_out)
    )
    plt.figure()
    shap.plots.waterfall(exp, show=False)
    plt.title(f"SHAP Waterfall — {label}")
    plt.tight_layout()
    plt.show()

## Fairness Analysis

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Full feature set (baseline + sensitive attributes)
full_features = [
    "EdLevel", "MainBranch", "YearsCode", "YearsCodePro", "PreviousSalary",
    "Gender", "Age", "MentalHealth"
]

numeric_features_full = ["YearsCode", "YearsCodePro", "PreviousSalary"]
categorical_features_full = ["EdLevel", "MainBranch", "Gender", "Age", "MentalHealth"]

X_full = df[full_features].copy()
y_full = df["Employed"].copy()

# Build preprocessing pipeline
numeric_transformer_full = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

categorical_transformer_full = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor_full = ColumnTransformer(transformers=[
    ("num", numeric_transformer_full,   numeric_features_full),
    ("cat", categorical_transformer_full, categorical_features_full)
])

# Train/test split
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

# Keep sensitive columns aligned with the test split for fairlearn
sensitive_test = X_test_full[["Gender", "Age"]].copy()

# Fit full model
full_model = Pipeline(steps=[
    ("preprocessor", preprocessor_full),
    ("classifier",   LogisticRegression(max_iter=2000, random_state=42))
])

full_model.fit(X_train_full, y_train_full)

y_pred_full  = full_model.predict(X_test_full)
y_proba_full = full_model.predict_proba(X_test_full)[:, 1]

print("=== Full Model (Baseline + Sensitive Features) ===")
print(f"Accuracy : {accuracy_score(y_test_full, y_pred_full):.4f}")
print(f"F1 Score : {f1_score(y_test_full, y_pred_full):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test_full, y_proba_full):.4f}")

In [ ]:
from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    equalized_odds_difference,
    selection_rate
)
from sklearn.metrics import accuracy_score

# MetricFrame broken down by Gender
mf_gender = MetricFrame(
    metrics={
        "accuracy":       accuracy_score,
        "selection_rate": selection_rate
    },
    y_true=y_test_full,
    y_pred=y_pred_full,
    sensitive_features=sensitive_test["Gender"]
)

print("=== Metrics by Gender ===")
print(mf_gender.by_group.to_string())
print(f"\nOverall accuracy      : {mf_gender.overall['accuracy']:.4f}")
print(f"Selection rate range  : {mf_gender.difference()['selection_rate']:.4f}")

# MetricFrame broken down by Age
mf_age = MetricFrame(
    metrics={
        "accuracy":       accuracy_score,
        "selection_rate": selection_rate
    },
    y_true=y_test_full,
    y_pred=y_pred_full,
    sensitive_features=sensitive_test["Age"]
)

print("\n=== Metrics by Age ===")
print(mf_age.by_group.to_string())
print(f"\nOverall accuracy      : {mf_age.overall['accuracy']:.4f}")
print(f"Selection rate range  : {mf_age.difference()['selection_rate']:.4f}")

In [ ]:
# Scalar fairness metrics
dp_gender = demographic_parity_difference(
    y_true=y_test_full,
    y_pred=y_pred_full,
    sensitive_features=sensitive_test["Gender"]
)

eo_gender = equalized_odds_difference(
    y_true=y_test_full,
    y_pred=y_pred_full,
    sensitive_features=sensitive_test["Gender"]
)

dp_age = demographic_parity_difference(
    y_true=y_test_full,
    y_pred=y_pred_full,
    sensitive_features=sensitive_test["Age"]
)

eo_age = equalized_odds_difference(
    y_true=y_test_full,
    y_pred=y_pred_full,
    sensitive_features=sensitive_test["Age"]
)

print("=== Fairness Summary ===")
print(f"{'Metric':<35} {'Gender':>10} {'Age':>10}")
print("-" * 57)
print(f"{'Demographic Parity Difference':<35} {dp_gender:>10.4f} {dp_age:>10.4f}")
print(f"{'Equalized Odds Difference':<35} {eo_gender:>10.4f} {eo_age:>10.4f}")
print("\n(0.0 = perfectly fair, 1.0 = maximally unfair)")

In [ ]:
# Visualize selection rate by group
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, mf, title in [
    (axes[0], mf_gender, "Selection Rate by Gender"),
    (axes[1], mf_age,    "Selection Rate by Age")
]:
    groups = mf.by_group.index.tolist()
    rates  = mf.by_group["selection_rate"].values
    overall = mf.overall["selection_rate"]

    bars = ax.bar(groups, rates, color="steelblue", edgecolor="white", width=0.5)
    ax.axhline(overall, color="tomato", linestyle="--", linewidth=1.5, label=f"Overall ({overall:.2f})")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_ylabel("Selection Rate (% predicted hired)")
    ax.set_ylim(0, 1)
    ax.legend()

    for bar, rate in zip(bars, rates):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.02,
                f"{rate:.2f}", ha="center", va="bottom", fontsize=10)

plt.suptitle("Fairlearn Bias Assessment", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Baseline vs Full model comparison
y_pred_baseline = log_model.predict(X_test)
y_proba_baseline = log_model.predict_proba(X_test)[:, 1]

print("=== Model Comparison ===")
print(f"{'Metric':<25} {'Baseline LR':>15} {'Full LR (+ sensitive)':>22}")
print("-" * 64)
print(f"{'Accuracy':<25} {accuracy_score(y_test, y_pred_baseline):>15.4f} {accuracy_score(y_test_full, y_pred_full):>22.4f}")
print(f"{'F1 Score':<25} {f1_score(y_test, y_pred_baseline):>15.4f} {f1_score(y_test_full, y_pred_full):>22.4f}")
print(f"{'ROC-AUC':<25} {roc_auc_score(y_test, y_proba_baseline):>15.4f} {roc_auc_score(y_test_full, y_proba_full):>22.4f}")
print(f"{'DP Diff (Gender)':<25} {'N/A':>15} {dp_gender:>22.4f}")
print(f"{'EO Diff (Gender)':<25} {'N/A':>15} {eo_gender:>22.4f}")
print(f"{'DP Diff (Age)':<25} {'N/A':>15} {dp_age:>22.4f}")
print(f"{'EO Diff (Age)':<25} {'N/A':>15} {eo_age:>22.4f}")
print("\nKey finding: Adding sensitive features provides minimal accuracy improvement")
print("but introduces measurable bias, particularly across gender groups.")

## Conclusions

### Key Findings:

1. **Model Performance**: Our logistic regression model achieves solid predictive performance using professional experience and salary information as the primary drivers of hiring predictions.

2. **Most Important Features**:
   - PreviousSalary and years of coding experience (YearsCode, YearsCodePro) are the strongest predictors
   - Education level and professional status (MainBranch) provide additional signal
   - Feature importance is distributed reasonably across multiple relevant factors

3. **Model Selection**: Logistic regression outperforms Random Forest in terms of generalization, with better train-test consistency and strong interpretability.

4. **Fairness Considerations**: Including sensitive demographic attributes (gender, age, mental health) in the model provides minimal accuracy gains but introduces measurable bias, particularly for gender. Our baseline model without these features is both fairer and more appropriate for real-world deployment.

### Recommendations:

- **Use the baseline logistic regression model** (without sensitive attributes) for resume screening applications
- Focus on objective professional metrics: experience levels and prior compensation
- Continue monitoring for fairness across demographic groups even when not explicitly modeling these features
- Consider this as a decision support tool rather than an automated screening system